In [ ]:
import harpy as hp
import spatialdata as sd
import matplotlib.pyplot as plt
import os

Specify the base path, where the output data is. In that folder, subfolders with regions and detected_transcripts.csv and images should exist.

In [ ]:
base_path = "path/to/merscope/output" # Change to your path

Load base and output path from config.py

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("../"))

from config import base_path

Load all regions in the folder of choice using harpy

In [ ]:
import spatialdata as sd
import harpy as hp
import os

analysis_path = base_path.replace("/output/", "/analysis/")
os.makedirs(analysis_path, exist_ok=True)

# Auto-detect regions
regions = sorted([
    d for d in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, d)) and d.startswith("region_")
])
print(f"Found regions: {regions}")

sdata_dict = {}

for region in regions:
    zarr_path = f"{analysis_path}/{region}.zarr"

    if os.path.exists(zarr_path):
        print(f"Loading {region} from zarr...")
        sdata_dict[region] = sd.read_zarr(zarr_path)
    else:
        print(f"Reading {region} from raw data...")
        sdata_dict[region] = hp.io.merscope(
            path=f"{base_path}/{region}",
            output=zarr_path,
            transcripts=True,
            cell_boundaries=False,
            rasterize_cell_boundaries=False,
            table=False,
            mosaic_images=False,
        )
    print(f"{region}: done")

print("All regions loaded")

For every zarr file, load polyT Z3 image (avoid adding all images per ZARR as each takes ~30GB of space in storage)

In [ ]:
import rioxarray
from spatialdata.models import Image2DModel
from spatialdata.transformations import Identity

channel = "PolyT"
z_layer = 3
mosaic = f"mosaic_{channel}_z{z_layer}"

for region in regions:
    # Skip if already added
    if mosaic in sdata_dict[region].images:
        print(f"{region}: mosaic already present, skipping")
        continue

    print(f"{region}: adding mosaic...")
    polyt = rioxarray.open_rasterio(
        f"{base_path}/{region}/images/{mosaic}.tif",
        chunks=(1, 4096, 4096), lock=False
    ).rename({"band": "c"})

    sdata_dict[region][mosaic] = Image2DModel.parse(
        polyt,
        dims=("c", "y", "x"),
        transformations={"global": Identity()},
    )
    sdata_dict[region].write_element(mosaic)
    del polyt
    print(f"{region}: done")

print("Mosaic added to all regions")

Move on to notebook 02_generate_plots_allregions.ipynb to generate overview plots, locate the sections in the mosaic images and generate plots per gene per root section